In [1]:
from datasets import load_dataset

In [84]:
train_data_file = "./data/when2call_train_sft.jsonl"
# split = "train"
# data = load_dataset("json", train_data_file)

In [3]:
import json
import jsonlines

In [4]:
with jsonlines.open(train_data_file, mode='r') as reader:
    data = reader.read()

In [86]:
with open(train_data_file, "r") as f:
    full_data = [json.loads(line) for line in f]

print(len(full_data))

15000


In [5]:
print(type(data))

<class 'dict'>


In [6]:
data.keys()

dict_keys(['tools', 'messages'])

In [7]:
print(data['tools'][0])

{"name": "get_stations_within_1_km", "description": "Fetch the nearest EV charging stations within a 1 km radius from a given latitude and longitude.", "parameters": {"type": "dict", "properties": {"region": {"description": "The region code (us for United States, ca for Canada, uk for United Kingdom, nz for New Zealand, hk for Hong Kong).", "type": "str", "default": ""}, "latitude": {"description": "The latitude of the location for which to find nearby charging stations.", "type": "int", "default": "40.733"}, "longitude": {"description": "The longitude of the location for which to find nearby charging stations.", "type": "int", "default": "-74.202"}}}, "required": ["region", "latitude", "longitude"]}


In [8]:
type(data['tools'][0])

str

In [9]:
tool = json.loads(data['tools'][0])

In [10]:
print(json.dumps(tool, indent=4))

{
    "name": "get_stations_within_1_km",
    "description": "Fetch the nearest EV charging stations within a 1 km radius from a given latitude and longitude.",
    "parameters": {
        "type": "dict",
        "properties": {
            "region": {
                "description": "The region code (us for United States, ca for Canada, uk for United Kingdom, nz for New Zealand, hk for Hong Kong).",
                "type": "str",
                "default": ""
            },
            "latitude": {
                "description": "The latitude of the location for which to find nearby charging stations.",
                "type": "int",
                "default": "40.733"
            },
            "longitude": {
                "description": "The longitude of the location for which to find nearby charging stations.",
                "type": "int",
                "default": "-74.202"
            }
        }
    },
    "required": [
        "region",
        "latitude",
        "longitu

In [11]:
type(data['messages'][0])

dict

In [12]:
print(data['messages'][0])

{'role': 'user', 'content': 'What are the trending topics in New York City today?'}


In [15]:
print(data)

{'tools': ['{"name": "get_stations_within_1_km", "description": "Fetch the nearest EV charging stations within a 1 km radius from a given latitude and longitude.", "parameters": {"type": "dict", "properties": {"region": {"description": "The region code (us for United States, ca for Canada, uk for United Kingdom, nz for New Zealand, hk for Hong Kong).", "type": "str", "default": ""}, "latitude": {"description": "The latitude of the location for which to find nearby charging stations.", "type": "int", "default": "40.733"}, "longitude": {"description": "The longitude of the location for which to find nearby charging stations.", "type": "int", "default": "-74.202"}}}, "required": ["region", "latitude", "longitude"]}'], 'messages': [{'role': 'user', 'content': 'What are the trending topics in New York City today?'}, {'role': 'assistant', 'content': "Apologies, but I'm unable to provide real-time information or perform web searches. You may want to check a reliable news source for that."}]}


In [17]:
from pprint import pprint
pprint(data["messages"])

[{'content': 'What are the trending topics in New York City today?',
  'role': 'user'},
 {'content': "Apologies, but I'm unable to provide real-time information or "
             'perform web searches. You may want to check a reliable news '
             'source for that.',
  'role': 'assistant'}]


In [88]:
# load xlam dataset
xlam_data_file = "./data/xlam_function_calling_60k.json"
with open(xlam_data_file, "r") as f:
    full_data = json.load(f)

In [96]:
full_data[20]

{'id': 20,
 'query': 'What is the binary sum of 10010 and 11101?',
 'answers': '[{"name": "binary_addition", "arguments": {"a": "10010", "b": "11101"}}]',
 'tools': '[{"name": "binary_addition", "description": "Adds two binary numbers and returns the result as a binary string.", "parameters": {"a": {"description": "The first binary number.", "type": "str"}, "b": {"description": "The second binary number.", "type": "str"}}}, {"name": "create_histogram", "description": "Create a histogram based on provided data.", "parameters": {"data": {"description": "The data for which histogram needs to be plotted.", "type": "list", "default": 10}, "bins": {"description": "The number of equal-width bins in the range. Default is 10.", "type": "int", "default": 10}}}]'}

In [95]:
from ast import literal_eval
type(full_data[0]["answers"]), type(literal_eval(full_data[0]["tools"])[0])

(str, dict)

In [98]:
abc = literal_eval(full_data[0]["answers"])
print(type(abc[0]))

<class 'dict'>


In [35]:
tools = [json.loads(tool) for tool in data["tools"]]

In [18]:
from transformers import AutoModel, AutoTokenizer

In [24]:
model_id = "google/gemma-3-270m-it"
model = AutoModel.from_pretrained(model_id)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

In [25]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [36]:
out = tokenizer.apply_chat_template(conversation=data["messages"], tools=tools, return_assistant_tokens_mask=True)
print(out)

{'input_ids': [2, 105, 2364, 107, 3689, 659, 506, 60194, 14050, 528, 1799, 3773, 4085, 3124, 236881, 106, 107, 105, 4368, 107, 22342, 7378, 236764, 840, 564, 236789, 236757, 13327, 531, 2847, 1759, 236772, 2289, 1938, 653, 2121, 4108, 34369, 236761, 1599, 1149, 1461, 531, 2426, 496, 13451, 4668, 3738, 573, 600, 236761, 106, 107], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'assistant_masks': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [39]:
print(len(out["input_ids"]), len(out["attention_mask"]), len(out["assistant_masks"]))

53 53 53


In [42]:
for item in out["input_ids"]:
    print(item, "\t", tokenizer.decode(item))

2 	 <bos>
105 	 <start_of_turn>
2364 	 user
107 	 

3689 	 What
659 	  are
506 	  the
60194 	  trending
14050 	  topics
528 	  in
1799 	  New
3773 	  York
4085 	  City
3124 	  today
236881 	 ?
106 	 <end_of_turn>
107 	 

105 	 <start_of_turn>
4368 	 model
107 	 

22342 	 Ap
7378 	 ologies
236764 	 ,
840 	  but
564 	  I
236789 	 '
236757 	 m
13327 	  unable
531 	  to
2847 	  provide
1759 	  real
236772 	 -
2289 	 time
1938 	  information
653 	  or
2121 	  perform
4108 	  web
34369 	  searches
236761 	 .
1599 	  You
1149 	  may
1461 	  want
531 	  to
2426 	  check
496 	  a
13451 	  reliable
4668 	  news
3738 	  source
573 	  for
600 	  that
236761 	 .
106 	 <end_of_turn>
107 	 



In [44]:
pprint(tokenizer.chat_template)

('{{ bos_token }}\n'
 "{%- if messages[0]['role'] == 'system' -%}\n"
 "    {%- if messages[0]['content'] is string -%}\n"
 "        {%- set first_user_prefix = messages[0]['content'] + '\n"
 '\n'
 "' -%}\n"
 '    {%- else -%}\n'
 "        {%- set first_user_prefix = messages[0]['content'][0]['text'] + '\n"
 '\n'
 "' -%}\n"
 '    {%- endif -%}\n'
 '    {%- set loop_messages = messages[1:] -%}\n'
 '{%- else -%}\n'
 '    {%- set first_user_prefix = "" -%}\n'
 '    {%- set loop_messages = messages -%}\n'
 '{%- endif -%}\n'
 '{%- for message in loop_messages -%}\n'
 "    {%- if (message['role'] == 'user') != (loop.index0 % 2 == 0) -%}\n"
 '        {{ raise_exception("Conversation roles must alternate '
 'user/assistant/user/assistant/...") }}\n'
 '    {%- endif -%}\n'
 "    {%- if (message['role'] == 'assistant') -%}\n"
 '        {%- set role = "model" -%}\n'
 '    {%- else -%}\n'
 "        {%- set role = message['role'] -%}\n"
 '    {%- endif -%}\n'
 "    {{ '<start_of_turn>' + role + '\n"

In [ ]:
with open("custom_gemma3_template.jinja2", "w") as f:
     f.write(tokenizer.chat_template)

In [47]:
tokenizer.save_pretrained("./gemma3_it-base")

('./gemma3_it-base/tokenizer_config.json',
 './gemma3_it-base/chat_template.jinja',
 './gemma3_it-base/tokenizer.json')

In [48]:
!ls

262_10968_92-pycharm-support-libs data_prep_func.py
data                              fine_tune.ipynb
data_exploration.ipynb            gemma3_it-base


In [55]:
tokenizer2 = AutoTokenizer.from_pretrained("./gemma3_it-base")

In [74]:
# load new tokenizer

In [99]:
import gc

del tokenizer2

gc.collect()

2365

In [100]:
tokenizer2 = AutoTokenizer.from_pretrained("./gemma3_it-base")

In [101]:
out = tokenizer2.apply_chat_template(conversation=data["messages"], tools=tools, return_assistant_tokens_mask=True)
print(out)

{'input_ids': [2, 105, 9731, 107, 3048, 659, 13474, 147224, 236764, 4464, 684, 40558, 9669, 4555, 236761, 1599, 659, 496, 11045, 16326, 236761, 108, 236865, 25373, 108, 3048, 1149, 2246, 886, 653, 919, 5151, 531, 6361, 607, 506, 2430, 7609, 768, 611, 75828, 625, 4127, 236761, 108, 3048, 659, 3847, 607, 1292, 42455, 2351, 655, 18220, 2588, 18220, 236813, 27161, 16616, 236787, 107, 236820, 18220, 236813, 107, 14937, 1201, 1083, 623, 828, 236779, 63817, 236779, 49426, 236779, 236770, 236779, 11375, 827, 623, 7777, 1083, 623, 30764, 506, 20480, 15620, 19416, 14996, 2351, 496, 236743, 236770, 7251, 12425, 699, 496, 2238, 40899, 532, 50550, 10498, 623, 19031, 1083, 3714, 2084, 1083, 623, 3074, 827, 623, 15921, 1083, 3714, 13647, 1083, 3714, 7777, 1083, 623, 818, 4128, 3393, 568, 605, 573, 3640, 4184, 236764, 2146, 573, 7203, 236764, 14378, 573, 3640, 15305, 236764, 136599, 573, 1799, 17564, 236764, 66669, 573, 15303, 14974, 769, 827, 623, 2084, 1083, 623, 1714, 827, 623, 6665, 1083, 3679, 12

In [60]:
for idx in range(len(out["input_ids"])):
    print(idx, "\t", out["input_ids"][idx], "\t", tokenizer.decode(out["input_ids"][idx]), "\t", out["assistant_masks"][idx])

0 	 2 	 <bos> 	 0
1 	 105 	 <start_of_turn> 	 0
2 	 2364 	 user 	 0
3 	 107 	 
 	 0
4 	 3689 	 What 	 0
5 	 659 	  are 	 0
6 	 506 	  the 	 0
7 	 60194 	  trending 	 0
8 	 14050 	  topics 	 0
9 	 528 	  in 	 0
10 	 1799 	  New 	 0
11 	 3773 	  York 	 0
12 	 4085 	  City 	 0
13 	 3124 	  today 	 0
14 	 236881 	 ? 	 0
15 	 106 	 <end_of_turn> 	 0
16 	 107 	 
 	 0
17 	 236743 	   	 1
18 	 105 	 <start_of_turn> 	 1
19 	 4368 	 model 	 1
20 	 107 	 
 	 1
21 	 10496 	  Ap 	 1
22 	 7378 	 ologies 	 0
23 	 236764 	 , 	 0
24 	 840 	  but 	 0
25 	 564 	  I 	 0
26 	 236789 	 ' 	 0
27 	 236757 	 m 	 0
28 	 13327 	  unable 	 0
29 	 531 	  to 	 0
30 	 2847 	  provide 	 0
31 	 1759 	  real 	 0
32 	 236772 	 - 	 0
33 	 2289 	 time 	 0
34 	 1938 	  information 	 0
35 	 653 	  or 	 0
36 	 2121 	  perform 	 0
37 	 4108 	  web 	 0
38 	 34369 	  searches 	 0
39 	 236761 	 . 	 0
40 	 1599 	  You 	 0
41 	 1149 	  may 	 0
42 	 1461 	  want 	 0
43 	 531 	  to 	 0
44 	 2426 	  check 	 0
45 	 496 	  a 	 0
46 	 1

In [58]:
import numpy as np
def find_x1_x2_next_x3(arr, x1, x2, x3):
    arr = np.asarray(arr)

    # Locations where x1, x2 occur consecutively
    starts = np.flatnonzero((arr[:-1] == x1) & (arr[1:] == x2))

    # Locations of x3
    x3_idx = np.flatnonzero(arr == x3)

    # For each start, find first x3 occurring after i+1
    pos = np.searchsorted(x3_idx, starts + 2)

    valid = pos < len(x3_idx)

    return list(zip(starts[valid], x3_idx[pos[valid]]))

In [59]:
find_x1_x2_next_x3(out["input_ids"], 105, 4368, 106)

[(np.int64(18), np.int64(52))]

In [67]:
def get_assistant_mask(input_ids, turn_start_id, assistant_token_id, turn_end_id):
    pairs = find_x1_x2_next_x3(input_ids, turn_start_id, assistant_token_id, turn_end_id)
    n = len(input_ids)
    assistant_mask = np.zeros(n, dtype=int)
    for i, j in pairs:
        assistant_mask[i+2:j+1] = 1

    return assistant_mask

In [102]:
assistant_mask = get_assistant_mask(out["input_ids"], 105, 4368, 106)

In [103]:
for idx in range(len(out["input_ids"])):
    print(idx, "\t", out["input_ids"][idx], "\t", tokenizer2.decode(out["input_ids"][idx]), "\t", assistant_mask[idx])

0 	 2 	 <bos> 	 0
1 	 105 	 <start_of_turn> 	 0
2 	 9731 	 system 	 0
3 	 107 	 
 	 0
4 	 3048 	 You 	 0
5 	 659 	  are 	 0
6 	 13474 	  Tool 	 0
7 	 147224 	  Gemma 	 0
8 	 236764 	 , 	 0
9 	 4464 	  created 	 0
10 	 684 	  by 	 0
11 	 40558 	  Trou 	 0
12 	 9669 	 blem 	 0
13 	 4555 	 aker 	 0
14 	 236761 	 . 	 0
15 	 1599 	  You 	 0
16 	 659 	  are 	 0
17 	 496 	  a 	 0
18 	 11045 	  helpful 	 0
19 	 16326 	  assistant 	 0
20 	 236761 	 . 	 0
21 	 108 	 

 	 0
22 	 236865 	 # 	 0
23 	 25373 	  Tools 	 0
24 	 108 	 

 	 0
25 	 3048 	 You 	 0
26 	 1149 	  may 	 0
27 	 2246 	  call 	 0
28 	 886 	  one 	 0
29 	 653 	  or 	 0
30 	 919 	  more 	 0
31 	 5151 	  functions 	 0
32 	 531 	  to 	 0
33 	 6361 	  assist 	 0
34 	 607 	  with 	 0
35 	 506 	  the 	 0
36 	 2430 	  user 	 0
37 	 7609 	  query 	 0
38 	 768 	  if 	 0
39 	 611 	  you 	 0
40 	 75828 	  deem 	 0
41 	 625 	  it 	 0
42 	 4127 	  necessary 	 0
43 	 236761 	 . 	 0
44 	 108 	 

 	 0
45 	 3048 	 You 	 0
46 	 659 	  are 	 0
47 	 

In [68]:
tool_calling_tokens = ["<tools>", "</tools>", "<tool_call>", "</tool_call>"]

for item in tool_calling_tokens:
    print(tokenizer.encode(item))

[2, 236820, 18220, 236813]
[2, 954, 18220, 236813]
[2, 236820, 13205, 236779, 6639, 236813]
[2, 954, 13205, 236779, 6639, 236813]


In [69]:
# add tool calling tokens to the tokenizer
tool_calling_tokens = ["<tools>", "</tools>", "<tool_call>", "</tool_call>"]
special_tokens = {
    "additional_special_tokens": tool_calling_tokens
}

num_added = tokenizer2.add_special_tokens(special_tokens)

_ = model.resize_token_embeddings(len(tokenizer2))

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Gemma3TextScaledWordEmbedding(262149, 640, padding_idx=0)

In [70]:
tool_calling_tokens = ["<tools>", "</tools>", "<tool_call>", "</tool_call>"]



[2, 262145]
[2, 262146]
[2, 262147]
[2, 262148]


In [73]:
print(tokenizer2.decode(262146))

</tools>
